In [0]:

# ==========================================
# 02 - BRONZE INGESTION
# Fraud Detection Lakehouse
# ==========================================

from pyspark.sql import functions as F

# Project configuration
CATALOG_NAME = "fraud_detection"
SOURCE_SCHEMA = "default"
SOURCE_VOLUME = "raw_data"
TARGET_SCHEMA = "bronze"

RAW_PATH = (
    f"/Volumes/{CATALOG_NAME}/"
    f"{SOURCE_SCHEMA}/{SOURCE_VOLUME}"
)

# Set active catalog
spark.sql(f"USE CATALOG {CATALOG_NAME}")

print(f"Source: {RAW_PATH}")
print(f"Destination: {CATALOG_NAME}.{TARGET_SCHEMA}")


In [0]:

# ------------------------------------------
# Reusable Bronze ingestion function
# ------------------------------------------

def ingest_to_bronze(dataset_name):

    source_path = f"{RAW_PATH}/{dataset_name}"
    target_table = f"{CATALOG_NAME}.{TARGET_SCHEMA}.{dataset_name}"

    print(f"Starting ingestion: {dataset_name}")

    # Read raw CSV data without automatically
    # inferring or converting source data types.
    df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", False)
        .load(source_path)
    )

    # Add ingestion metadata
    df = (
        df
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_ingestion_date", F.current_date())
    )

    # Save as a managed Delta table
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )

    print(f"Successfully loaded: {target_table}")
    print(f"Records ingested: {df.count():,}")

    return target_table


In [0]:
# ------------------------------------------
# Ingest all source datasets
# ------------------------------------------

DATASETS = [
    "customers",
    "merchants",
    "transactions"
]

for dataset in DATASETS:
    ingest_to_bronze(dataset)

print("Bronze ingestion completed!")

In [0]:
# ------------------------------------------
# Validate Bronze ingestion
# ------------------------------------------

for dataset in DATASETS:

    table_name = f"{CATALOG_NAME}.{TARGET_SCHEMA}.{dataset}"

    df = spark.table(table_name)

    print(f"\nDataset: {table_name}")
    print(f"Record count: {df.count():,}")

    display(df.limit(5))